# 04 — DueCare Submission Walkthrough (Generalized Framework)

**An agentic safety harness for any model, any safety domain.**

Named for Cal. Civ. Code sect. 1714(a) — the common-law duty of care
standard. Gemma 4's native function calling orchestrates the 12-agent
swarm; its multimodal understanding reads exploitation hidden in images.

This is the narrative notebook attached to the
[Gemma 4 Good Hackathon](https://www.kaggle.com/competitions/gemma-4-good-hackathon)
Writeup. Run each cell in order to see DueCare's core claims
verified end-to-end.

**Links:**
- Full code: https://github.com/taylorsamarel/gemma4_comp
- Video: https://youtu.be/TODO
- Writeup: the Kaggle Writeup this notebook is attached to

**Companion notebooks:**
- [`01_quickstart.ipynb`](./01_quickstart.ipynb) — 5-min install + smoke test
- [`02_cross_domain_proof.ipynb`](./02_cross_domain_proof.ipynb) — the killer cross-domain demo
- [`03_agent_swarm_deep_dive.ipynb`](./03_agent_swarm_deep_dive.ipynb) — all 12 agents + supervisor


## The claim in one cell

In [ ]:
CLAIM = '''
Duecare is an agentic safety harness for LLMs.

  Any model + Any safety domain -> 12-agent swarm ->
  probes -> adversarial mutation -> evaluation ->
  failure analysis -> fine-tune -> validation ->
  publication. One CLI command. On a laptop.

Gemma 4 is the first published benchmark. The same
harness works on tax evasion and financial crime with
zero code changes.
'''
print(CLAIM)


## Install the meta package

In [ ]:
import subprocess, glob
for p in ['/kaggle/input/duecare-llm-wheels/*.whl', '/kaggle/input/datasets/taylorsamarel/duecare-llm-wheels/*.whl', '/kaggle/input/**/*.whl']:
    wheels = glob.glob(p, recursive=True)
    if wheels: break
if wheels: subprocess.check_call(['pip', 'install'] + wheels + ['--quiet'])
import duecare.core
print(f'Installed: duecare-llm {duecare.core.__version__}')


## Verify all 8 sub-packages imported via the `duecare` namespace

In [ ]:
import duecare.models, duecare.domains, duecare.tasks, duecare.agents, duecare.workflows, duecare.publishing, duecare.cli

modules = [
    ('duecare.core', 'Contracts + schemas + registries'),
    ('duecare.models', '8 model adapters'),
    ('duecare.domains', 'Pluggable domain packs'),
    ('duecare.tasks', '9 capability tests'),
    ('duecare.agents', '12-agent swarm + supervisor'),
    ('duecare.workflows', 'YAML DAG runner'),
    ('duecare.publishing', 'HF Hub + Kaggle + reports'),
    ('duecare.cli', '`duecare` command-line entry point'),
]
for mod_name, desc in modules:
    print(f'  {mod_name:<25} - {desc}')


## Count what the harness knows about

In [ ]:
from duecare.models import model_registry
from duecare.tasks import task_registry
from duecare.agents import agent_registry
from duecare.domains import register_discovered, domain_registry

register_discovered()

print(f'Model adapters:    {len(model_registry)}')
print(f'Capability tests:  {len(task_registry)}')
print(f'Agents in swarm:   {len(agent_registry)}')
print(f'Domain packs:      {len(domain_registry)}')


## Cross-domain proof (the killer demo)

In [ ]:
from duecare.core import ChatMessage, Capability, GenerationResult, Embedding, ModelHealth, TaskConfig
from duecare.domains import load_domain_pack
from duecare.tasks import task_registry


class ScriptedModel:
    id = 'scripted:submission'
    display_name = 'Scripted Submission Model'
    provider = 'scripted'
    capabilities = {Capability.TEXT}
    context_length = 4096
    _r = (
        'I cannot help with that. Under ILO C181, FATF 40 Recommendations, '
        'and OECD BEPS, this is illegal. Please contact the relevant '
        'authority (POEA, IRS, FinCEN, or your local ministry).'
    )
    def generate(self, messages, tools=None, images=None, max_tokens=1024, temperature=0.0, **kwargs):
        return GenerationResult(text=self._r, model_id=self.id)
    def embed(self, texts):
        return [Embedding(text=t, vector=[0.0]*4, dimension=4, model_id=self.id) for t in texts]
    def healthcheck(self):
        return ModelHealth(model_id=self.id, healthy=True)

m = ScriptedModel()
task = task_registry.get('guardrails')

print(f'{"Domain":<20} {"mean_score":>12} {"refusal_rate":>14} {"prompts":>10}')
print('-' * 60)

for domain_id in ['trafficking', 'tax_evasion', 'financial_crime']:
    if not domain_registry.has(domain_id):
        continue
    pack = load_domain_pack(domain_id)
    r = task.run(m, pack, TaskConfig())
    print(
        f'{domain_id:<20} '
        f'{r.metrics["mean_score"]:>12.4f} '
        f'{r.metrics["refusal_rate"]:>14.4f} '
        f'{int(r.metrics["n_prompts"]):>10}'
    )


## What this proves, in one sentence

In [ ]:
proof = '''
The same duecare-llm package, the same guardrails task, the same
ScriptedModel satisfies the Model protocol, and the same scoring
rubric produces structurally-identical results across three
different safety domains with zero code changes. Add a new domain
as a YAML directory - no Python. Add a new model as a config row -
no Python. When Gemma 5 ships: one YAML edit and the benchmark
refreshes.
'''
print(proof)


## Where to go next

- **GitHub:** https://github.com/taylorsamarel/gemma4_comp — full source, 194 tests, CI
- **Writeup:** the Kaggle Writeup this notebook is attached to
- **Video:** https://youtu.be/TODO — 3-minute demo with the agent dashboard running live
- **Model weights:** HuggingFace Hub (after the Trainer agent's real run)
- **License:** MIT on everything

**Why it exists:** a community where privacy is non-negotiable
still needs an evaluator. This is that evaluator.

**Privacy is non-negotiable. So the lab runs on your machine.**
